In [1]:
import json
import pandas as pd
import numpy as np

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
print(train.shape, test.shape)

(10003, 2) (3080, 2)


In [2]:
DEPARTMENTS = {
    "Card Services": [
        "activate_my_card", "card_arrival", "card_delivery_estimate", "order_physical_card",
        "get_physical_card", "getting_spare_card", "getting_virtual_card",
        "get_disposable_virtual_card", "disposable_card_limits", "virtual_card_not_working",
        "card_not_working", "contactless_not_working", "card_about_to_expire", "card_linking",
        "card_acceptance", "card_swallowed", "visa_or_mastercard", "apple_pay_or_google_pay",
        "change_pin", "pin_blocked"],
    "Fraud & Security": [
        "compromised_card", "lost_or_stolen_card", "lost_or_stolen_phone",
        "card_payment_not_recognised", "cash_withdrawal_not_recognised",
        "direct_debit_payment_not_recognised"],
    "Payments & Cash": [
        "pending_card_payment", "declined_card_payment", "reverted_card_payment?",
        "transaction_charged_twice", "wrong_amount_of_cash_received",
        "declined_cash_withdrawal", "pending_cash_withdrawal", "atm_support"],
    "Transfers": [
        "cancel_transfer", "beneficiary_not_allowed", "pending_transfer", "failed_transfer",
        "declined_transfer", "transfer_timing", "transfer_not_received_by_recipient",
        "balance_not_updated_after_bank_transfer", "transfer_into_account", "receiving_money"],
    "Top-ups": [
        "pending_top_up", "top_up_reverted", "top_up_failed", "automatic_top_up",
        "verify_top_up", "top_up_by_cash_or_cheque", "top_up_by_card_charge",
        "top_up_by_bank_transfer_charge", "topping_up_by_card", "top_up_limits",
        "balance_not_updated_after_cheque_or_cash_deposit"],
    "Billing, Fees & Refunds": [
        "card_payment_fee_charged", "cash_withdrawal_charge", "transfer_fee_charged",
        "extra_charge_on_statement", "request_refund", "Refund_not_showing_up",
        "exchange_charge", "exchange_rate", "exchange_via_app",
        "card_payment_wrong_exchange_rate", "wrong_exchange_rate_for_cash_withdrawal"],
    "Account & Verification": [
        "why_verify_identity", "verify_my_identity", "unable_to_verify_identity",
        "verify_source_of_funds", "edit_personal_details", "terminate_account", "age_limit",
        "passcode_forgotten", "country_support", "supported_cards_and_currencies",
        "fiat_currency_support"],
}

intent_to_dept = {i: d for d, intents in DEPARTMENTS.items() for i in intents}

all_intents = set(train["category"])
print("Mapped:", len(intent_to_dept), "/", len(all_intents))
print("Missing:", all_intents - set(intent_to_dept))
print("Unknown:", set(intent_to_dept) - all_intents)

train["department"] = train["category"].map(intent_to_dept)
test["department"] = test["category"].map(intent_to_dept)

with open("../models/departments.json", "w") as f:
    json.dump(intent_to_dept, f, indent=2)

Mapped: 77 / 77
Missing: set()
Unknown: set()


In [3]:
train["department"].value_counts()

department
Card Services              2169
Billing, Fees & Refunds    1714
Top-ups                    1413
Transfers                  1409
Account & Verification     1268
Payments & Cash            1231
Fraud & Security            799
Name: count, dtype: int64

In [4]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(name, true_intents, pred_intents):
    true_d = pd.Series(true_intents).map(intent_to_dept)
    pred_d = pd.Series(pred_intents).map(intent_to_dept)
    print(f"{name:24s} | intent acc: {accuracy_score(true_intents, pred_intents)*100:5.1f}% "
          f"| intent macro-F1: {f1_score(true_intents, pred_intents, average='macro')*100:5.1f}% "
          f"| department acc: {accuracy_score(true_d, pred_d)*100:5.1f}%")

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000, C=10)),
])

tfidf_lr.fit(train["text"], train["category"])
evaluate("TF-IDF + LogReg", test["category"], tfidf_lr.predict(test["text"]))

TF-IDF + LogReg          | intent acc:  89.1% | intent macro-F1:  89.1% | department acc:  95.9%


In [6]:
pred = tfidf_lr.predict(test["text"])
wrong = test.assign(pred=pred).query("category != pred")
print("Wrong:", len(wrong), "of", len(test))
pd.Series(list(zip(wrong["category"], wrong["pred"]))).value_counts().head(10)

Wrong: 337 of 3080


(pin_blocked, get_physical_card)                                                 5
(unable_to_verify_identity, verify_my_identity)                                  5
(verify_my_identity, why_verify_identity)                                        5
(fiat_currency_support, exchange_via_app)                                        4
(pending_top_up, top_up_failed)                                                  4
(card_acceptance, card_not_working)                                              4
(top_up_reverted, top_up_failed)                                                 4
(why_verify_identity, verify_my_identity)                                        4
(wrong_exchange_rate_for_cash_withdrawal, card_payment_wrong_exchange_rate)      4
(balance_not_updated_after_bank_transfer, transfer_not_received_by_recipient)    4
Name: count, dtype: int64

### Baseline results: TF-IDF + Logistic Regression
| Metric | Score |
|---|---|
| Intent accuracy | 89.1% |
| Intent macro-F1 | 89.1% |
| Department accuracy | **95.9%** |

- Macro-F1 equals accuracy, so the model performs evenly across small and large intents.
- Most mistakes are between intents that mean almost the same thing (e.g. `verify_my_identity` vs `why_verify_identity`).
- 7 of the 10 most common mistakes stay inside the same department, which is why department accuracy (95.9%) is much higher than intent accuracy (89.1%).


In [7]:
from sentence_transformers import SentenceTransformer

encoders = {
    "MiniLM (small, fast)": "all-MiniLM-L6-v2",
    "MPNet (larger, more accurate)": "all-mpnet-base-v2",
}

embeddings = {}
for name, model_id in encoders.items():
    print("Encoding with", name)
    enc = SentenceTransformer(model_id)
    X_train = enc.encode(train["text"].tolist(), batch_size=64,
                         show_progress_bar=True, normalize_embeddings=True)
    X_test = enc.encode(test["text"].tolist(), batch_size=64,
                        show_progress_bar=True, normalize_embeddings=True)
    embeddings[name] = (X_train, X_test)
    print("  shape:", X_train.shape)

Encoding with MiniLM (small, fast)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

  shape: (10003, 384)
Encoding with MPNet (larger, more accurate)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

... (13 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

  shape: (10003, 768)


In [9]:
for name, (X_train, X_test) in embeddings.items():
    clf = LogisticRegression(max_iter=3000, C=10)
    clf.fit(X_train, train["category"])
    evaluate(f"{name.split()[0]} + LogReg", test["category"], clf.predict(X_test))

evaluate("TF-IDF + LogReg", test["category"], tfidf_lr.predict(test["text"]))

MiniLM + LogReg          | intent acc:  93.0% | intent macro-F1:  93.0% | department acc:  97.7%
MPNet + LogReg           | intent acc:  93.7% | intent macro-F1:  93.7% | department acc:  98.1%
TF-IDF + LogReg          | intent acc:  89.1% | intent macro-F1:  89.1% | department acc:  95.9%


### Results: word-matching vs meaning-based models
| Model | Intent accuracy | Macro-F1 | Department accuracy |
|---|---|---|---|
| TF-IDF + LogReg | 89.1% | 89.1% | 95.9% |
| MiniLM + LogReg | 93.0% | 93.0% | 97.7% |
| MPNet + LogReg | 93.7% | 93.7% | 98.1% |

- Pretrained sentence embeddings (transfer learning) beat word-matching by ~4 points, because they understand that different wordings can mean the same thing.
- **Chosen model: MiniLM.** It's within 0.7 points of MPNet but 5× smaller and faster, which matters for a free hosted web app.

In [10]:
X_train, X_test = embeddings["MiniLM (small, fast)"]

final_clf = LogisticRegression(max_iter=3000, C=10)
final_clf.fit(X_train, train["category"])

proba = final_clf.predict_proba(X_test)
pred = final_clf.classes_[proba.argmax(axis=1)]
confidence = proba.max(axis=1)

correct = pred == test["category"].values
print("Average confidence when RIGHT:", round(confidence[correct].mean(), 3))
print("Average confidence when WRONG:", round(confidence[~correct].mean(), 3))

Average confidence when RIGHT: 0.884
Average confidence when WRONG: 0.529


In [11]:
dept_names = list(DEPARTMENTS)
dept_of_class = np.array([intent_to_dept[c] for c in final_clf.classes_])

dept_proba = np.column_stack(
    [proba[:, dept_of_class == d].sum(axis=1) for d in dept_names])
dept_pred = np.array(dept_names)[dept_proba.argmax(axis=1)]
dept_conf = dept_proba.max(axis=1)
dept_true = test["department"].values

rows = []
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    auto = dept_conf >= t
    rows.append({
        "cut-off": t,
        "auto-routed %": auto.mean() * 100,
        "correct when auto-routed %": (dept_pred[auto] == dept_true[auto]).mean() * 100,
        "sent to human %": (~auto).mean() * 100,
    })
pd.DataFrame(rows).round(1)

,cut-off,auto-routed %,correct when auto-routed %,sent to human %
0,0.0,100.0,97.7,0.0
1,0.5,98.1,98.3,1.9
2,0.6,96.0,99.1,4.0
3,0.7,93.4,99.4,6.6
4,0.8,89.9,99.7,10.1
5,0.9,82.6,99.9,17.4


### Confidence-based routing ("if unsure, send to a human")
The model is a good judge of its own certainty: it averages **88% confidence when right** and **53% when wrong**.

Department confidence = the sum of the probabilities of all intents in that department.

| Cut-off | Auto-routed | Accuracy when auto-routed | Sent to human |
|---|---|---|---|
| None | 100% | 97.7% | 0% |
| **0.6 (chosen)** | **96.0%** | **99.1%** | **4.0%** |
| 0.8 | 89.9% | 99.7% | 10.1% |
| 0.9 | 82.6% | 99.9% | 17.4% |

**Chosen cut-off: 0.6.** 96% of messages are routed automatically with 99.1% accuracy, and only 4% need a human. Higher cut-offs gain little accuracy but greatly increase the human workload.